# Reusable Template: Regularized Regression for Economic Data

A parameterized notebook for any economics regression project — wage equations, housing price models, demand estimation, growth regressions. Edit only the config cell to point it at a new dataset.

**How to reuse this for a new project:**
1. Point `CSV_PATH` at your file, set `TARGET_COL`, and list any columns to exclude (e.g., a raw-level version of a log target, IDs, or the treatment variable if you're doing double-selection).
2. If your target is skewed (typical for wages, prices, firm size), set `LOG_TRANSFORM_TARGET = True`.
3. Run all cells top to bottom.
4. Widen `ALPHA_RANGE` if the tuned alpha lands at the edge of the grid.

Demonstrated below on `wage_survey.csv`, but nothing except the config cell is dataset-specific.

## 1. Configuration — edit this cell for your project

In [ ]:
CONFIG = {
    'CSV_PATH': 'wage_survey.csv',
    'TARGET_COL': 'log_hourly_wage',
    'EXCLUDE_COLS': ['hourly_wage'],   # other columns to drop besides the target (e.g. raw level of a logged target)
    'LOG_TRANSFORM_TARGET': False,     # set True if TARGET_COL is a raw level you want to log yourself
    'TEST_SIZE': 0.2,
    'RANDOM_STATE': 42,
    'ALPHA_RANGE': (-4, 4),            # exponents for np.logspace
    'N_GRID_POINTS': 100,
    'CV_FOLDS': 5,
    'L1_RATIOS': [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99],
}
CONFIG

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 3. Load & prepare data

In [ ]:
def load_data(config):
    df = pd.read_csv(config['CSV_PATH'])
    y = df[config['TARGET_COL']]
    if config['LOG_TRANSFORM_TARGET']:
        y = np.log(y)
    drop_cols = [config['TARGET_COL']] + config['EXCLUDE_COLS']
    X_raw = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return df, X_raw, y

def scale_and_split(X_raw, y, config):
    scaler = StandardScaler().fit(X_raw)
    X = scaler.transform(X_raw)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config['TEST_SIZE'], random_state=config['RANDOM_STATE']
    )
    return X_train, X_test, y_train, y_test, scaler

df, X_raw, y = load_data(CONFIG)
X_train, X_test, y_train, y_test, scaler = scale_and_split(X_raw, y, CONFIG)
predictors = X_raw.columns
print(df.shape, '-> train:', X_train.shape, 'test:', X_test.shape)

## 4. Baseline OLS — sanity check for overfitting / collinearity

In [ ]:
def evaluate(model, X, y, label=''):
    pred = model.predict(X)
    return {'label': label, 'MSE': mean_squared_error(y, pred),
            'MAE': mean_absolute_error(y, pred), 'R2': r2_score(y, pred)}

ols = LinearRegression().fit(X_train, y_train)
print(evaluate(ols, X_train, y_train, 'train'))
print(evaluate(ols, X_test, y_test, 'test'))

## 5. Regularized models with automatic alpha tuning

In [ ]:
def fit_all_regularized(X_train, y_train, config):
    alphas = np.logspace(*config['ALPHA_RANGE'], config['N_GRID_POINTS'])
    ridge_cv = RidgeCV(alphas=alphas, cv=config['CV_FOLDS']).fit(X_train, y_train)
    lasso_cv = LassoCV(alphas=alphas, cv=config['CV_FOLDS'], max_iter=10000).fit(X_train, y_train)
    enet_cv = ElasticNetCV(
        alphas=alphas, l1_ratio=config['L1_RATIOS'], cv=config['CV_FOLDS'], max_iter=10000
    ).fit(X_train, y_train)
    return {'Ridge': ridge_cv, 'Lasso': lasso_cv, 'ElasticNet': enet_cv}

models = fit_all_regularized(X_train, y_train, CONFIG)
for name, m in models.items():
    print(name, '-> alpha:', round(m.alpha_, 5),
          '| l1_ratio:', getattr(m, 'l1_ratio_', 'n/a'))

## 6. Evaluate on held-out test set

In [ ]:
rows = [dict(evaluate(ols, X_test, y_test, 'OLS'), **{'Nonzero coefs': len(predictors)})]
for name, m in models.items():
    row = evaluate(m, X_test, y_test, name)
    row['Nonzero coefs'] = int(np.sum(np.abs(m.coef_) > 1e-8))
    rows.append(row)

pd.DataFrame(rows).set_index('label').round(4)

## 7. Coefficient plot for the best model (lowest test MSE)

In [ ]:
best_name = min(models, key=lambda n: mean_squared_error(y_test, models[n].predict(X_test)))
best_model = models[best_name]
print('Best regularized model:', best_name)

coef = pd.Series(best_model.coef_, predictors).sort_values()
coef.plot(kind='bar', title=f'{best_name} coefficients (standardized)')
plt.axhline(0, color='k', linewidth=0.8)
plt.tight_layout()
plt.show()

## 8. Rescale coefficients back to original units (for write-ups)

In [ ]:
raw_coef = pd.Series(best_model.coef_ / scaler.scale_, predictors).sort_values()
raw_coef

If `TARGET_COL` is a log target, each `raw_coef` entry is approximately "percent change in the target per one-unit change in that feature, holding others fixed" (exact conversion: `100*(exp(coef)-1)`).

## 9. Adapting this template

- **New economic outcome:** change `CSV_PATH`/`TARGET_COL`; set `LOG_TRANSFORM_TARGET=True` if you're handing it a raw (unlogged) skewed outcome like price or income.
- **Panel/time-series data:** this template assumes independent cross-sectional observations. For panel data, add entity/time fixed effects as dummy columns before scaling, or use a dedicated panel-data package (e.g., `linearmodels`) instead of plain scikit-learn.
- **You care about one specific coefficient (policy variable), not just prediction:** don't report the Ridge/Lasso coefficient directly — use the double-selection pattern from the cheat sheet notebook (`econ_03`) and finish with plain OLS on the selected controls.
- **Classification instead of regression** (e.g., predicting loan default or labor-force participation): swap in `LogisticRegression`/`LogisticRegressionCV` and F1/ROC-AUC metrics — see the original (non-economics) reusable template for that branch.